In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from measurement_loader import (
    load_measurement,
    load_measurements,
    search_best_objective,
    magnitude_db,
    notch_depth_objective,
)

## Caricare e confrontare piu' misure

In [ ]:
# Esempio: sessione di Training o Exhaustive Search, con
# sottocartelle iter0, iter1, ... (vedi TrainingSession.session_folder)
#SESSION_FOLDER = Path("trainings/my_session")

#measurements = load_measurements(SESSION_FOLDER, pattern="iter*")
#print(f"Misure caricate: {len(measurements)}")

#Singola misura (es. dalla tab Single Measurement)
data = load_measurement(r"C:\Users\Fabrizio\OneDrive - Politecnico di Milano\Tesi\Neuromorphic\N2\N2 code\test_measurements\Test2")

In [ ]:
def plot_measurements(measurements, trace="S21", gated=True, ax=None):
    """Sovrappone lo spettro di piu' misure sullo stesso plot,
    etichettando ciascuna curva con il nome della sua cartella."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))

    for data in measurements:
        source = data.vna_results if gated else data.vna_results_raw
        if trace not in source:
            continue
        mag_db = magnitude_db(data, trace, gated=gated)
        label = data.folder.name if hasattr(data, "folder") else data.timestamp
        ax.plot(data.freq_axis / 1e9, mag_db, label=label)

    ax.set_xlabel("Frequency [GHz]")
    ax.set_ylabel(f"|{trace}| [dB]" + (" (gated)" if gated else " (raw)"))
    ax.legend(fontsize=8)
    ax.grid(True)
    return ax


plot_measurements(data, trace="S21")
plt.show()

In [ ]:
def plot_measurement(data, trace="S21", gated=True, ax=None):
    """Sovrappone lo spettro di piu' misure sullo stesso plot,
    etichettando ciascuna curva con il nome della sua cartella."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))

    mag_db = magnitude_db(data, trace, gated=gated)
    label = data.folder.name if hasattr(data, "folder") else data.timestamp
    ax.plot(data.freq_axis / 1e9, mag_db, label=label)

    ax.set_xlabel("Frequency [GHz]")
    ax.set_ylabel(f"|{trace}| [dB]" + (" (gated)" if gated else " (raw)"))
    ax.legend(fontsize=8)
    ax.grid(True)
    return ax


plot_measurement(data, trace="S21", gated = False)
plt.show()

## Ricerca dell'obiettivo migliore nel dataset

Esempio: per ogni misura, trova la frequenza di notch che massimizza la profondita' del notch (stesso spirito delle celle "NOTCH FILTER" sopra, ma sui dati sperimentali).

In [ ]:
candidate_freqs = np.arange(2.0, 2.5, 0.005)  # GHz

results = search_best_objective(
    measurements,
    objective_fn=notch_depth_objective,
    param_grid={"notch_freq": candidate_freqs},
)

for r in results[:5]:
    label = r["measurement"].folder.name
    print(f"{label:15s} score={r['score']:.2f} dB  params={r['params']}")

In [ ]:
# Confronto visivo delle 5 misure migliori secondo l'obiettivo
best_measurements = [r["measurement"] for r in results[:5]]
plot_measurements(best_measurements, trace="S21")
plt.show()

### Definire un objective diverso

Qualsiasi funzione con firma `objective_fn(data: MeasurementData, **params) -> float` puo' essere passata a `search_best_objective`. Esempio senza parametri liberi (punteggio pari alla magnitudine massima di S21 in banda passante):

In [ ]:
def passband_max_objective(data, band_start=2.0, band_stop=2.2, trace="S21"):
    freq_ghz = data.freq_axis / 1e9
    mag_db = magnitude_db(data, trace)
    mask = (freq_ghz >= band_start) & (freq_ghz <= band_stop)
    return float(np.max(mag_db[mask])) if np.any(mask) else -np.inf


results_passband = search_best_objective(measurements, objective_fn=passband_max_objective)
best = results_passband[0]
print(f"Migliore: {best['measurement'].folder.name}, score={best['score']:.2f} dB")